# ETL Pipeline Demonstration

This notebook demonstrates the Extract–Transform–Load (ETL) pipeline used in this project.

The purpose of this notebook is to:

- load the raw data;
- execute the preprocessing pipeline;
- demonstrate every transformation stage;
- validate the processed datasets;

---

## 1. Imports

In [1]:
from pathlib import Path

import pandas as pd

from src.data.etl import (
    ItemCategoryLoader,
    ItemsLoader,
    ShopsLoader,
    TrainLoader,
    DfFinalLoader,
    ForSubmissionLoader,
)

## 2. Project Structure

The ETL pipeline reads the raw CSV files located in

```
data/raw/
```

and stores processed datasets inside

```
data/processed/
```

Each dataset is generated independently and cached for future use.


In [2]:
project_root = Path.cwd().parent

raw_path = project_root / "data" / "raw"
processed_path = project_root / "data" / "processed"

print(raw_path)
print(processed_path)

c:\Users\user\predict-future-sales\data\raw
c:\Users\user\predict-future-sales\data\processed


## 3. Item Categories Processing

The item categories dataset is enriched with a new feature:

- **main_category** extracted from the original category name.


In [3]:
item_categories = ItemCategoryLoader(raw_data_path=raw_path, cache_path=processed_path).load()

item_categories.head()

,item_category_name,item_category_id,main_category
0,PC - Гарнитуры/Наушники,0,PC
1,Аксессуары - PS2,1,Аксессуары
2,Аксессуары - PS3,2,Аксессуары
3,Аксессуары - PS4,3,Аксессуары
4,Аксессуары - PSP,4,Аксессуары


In [4]:
item_categories.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 3 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   item_category_name  84 non-null     object
 1   item_category_id    84 non-null     int64 
 2   main_category       84 non-null     object
dtypes: int64(1), object(2)
memory usage: 2.1+ KB


## 4. Items Processing

The items dataset undergoes several preprocessing steps.

The pipeline:

- cleans item names;
- identifies duplicated products;
- assigns corrected item IDs.


In [5]:
items = ItemsLoader(raw_data_path=raw_path, cache_path=processed_path).load()

items.head()

,item_name,item_id,item_category_id,corrected_item_id
0,ВО ВЛАСТИ НАВАЖДЕНИЯ (ПЛАСТ.) Disc,0,40,0
1,ABBYY FineReader 12 Professional Edition Full ...,1,76,1
2,В ЛУЧАХ СЛАВЫ (UNV) Disc,2,40,2
3,ГОЛУБАЯ ВОЛНА (Univ) Disc,3,40,3
4,КОРОБКА (СТЕКЛО) Disc,4,40,4


In [6]:
items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22170 entries, 0 to 22169
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   item_name          22170 non-null  object
 1   item_id            22170 non-null  int64 
 2   item_category_id   22170 non-null  int64 
 3   corrected_item_id  22170 non-null  int64 
dtypes: int64(3), object(1)
memory usage: 692.9+ KB


## 5. Shops Processing

The shops dataset is cleaned and standardized.

Transformations include:

- cleaning shop names;
- assigning corrected IDs;
- assigning is shop a franchise;
- extracting city names;


In [7]:
shops = ShopsLoader(raw_data_path=raw_path, cache_path=processed_path).load()

shops.head()

,shop_name,shop_id,city,corrected_shop_id,is_fran
0,"Якутск Орджоникидзе, 56 фран",0,Якутск,57,0
1,"Якутск ТЦ ""Центральный"" фран",1,Якутск,58,0
2,"Адыгея ТЦ ""Мега""",2,Адыгея,2,0
3,"Балашиха ТРК ""Октябрь-Киномир""",3,Балашиха,3,0
4,"Волжский ТЦ ""Волга Молл""",4,Волжский,4,0


In [8]:
shops.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 60 entries, 0 to 59
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   shop_name          60 non-null     object
 1   shop_id            60 non-null     int64 
 2   city               60 non-null     object
 3   corrected_shop_id  60 non-null     int64 
 4   is_fran            60 non-null     int64 
dtypes: int64(3), object(2)
memory usage: 2.5+ KB


## 6. Sales Data Processing

The pipeline performs:

- duplicate removal;
- datetime parsing;
- extraction of temporal features;
- computation of inflation-adjusted prices.


In [9]:
train = TrainLoader(raw_data_path=raw_path, cache_path=processed_path).load()

train.head()

,date,date_block_num,shop_id,item_id,item_price,item_cnt_day,year,month,day,week_day,no_inflation_price
0,2013-01-02,0,59,22154,999.00,1.0,2013,1,2,2,989.402793
1,2013-01-03,0,25,2552,899.00,1.0,2013,1,3,3,890.363474
2,2013-01-05,0,25,2552,899.00,-1.0,2013,1,5,5,890.363474
3,2013-01-06,0,25,2554,1709.05,1.0,2013,1,6,6,1692.631475
4,2013-01-15,0,25,2555,1099.00,1.0,2013,1,15,1,1088.442112


In [10]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2935843 entries, 0 to 2935848
Data columns (total 11 columns):
 #   Column              Dtype         
---  ------              -----         
 0   date                datetime64[ns]
 1   date_block_num      int64         
 2   shop_id             int64         
 3   item_id             int64         
 4   item_price          float64       
 5   item_cnt_day        float64       
 6   year                int32         
 7   month               int32         
 8   day                 int32         
 9   week_day            int32         
 10  no_inflation_price  float64       
dtypes: datetime64[ns](1), float64(3), int32(4), int64(3)
memory usage: 224.0 MB


In [11]:
train.isna().sum()

date                  0
date_block_num        0
shop_id               0
item_id               0
item_price            0
item_cnt_day          0
year                  0
month                 0
day                   0
week_day              0
no_inflation_price    0
dtype: int64

## 7. Building the Final Dataset

The final dataset is produced by joining all processed datasets.

The following information is combined:

- sales history;
- item metadata;
- category information;
- shop information.

Additional manual corrections and data cleaning are also applied.


In [12]:
df = DfFinalLoader(cache_path=processed_path, shops=shops, items=items, item_cats=item_categories, train=train).load()

df.head()

,date,date_block_num,item_price,item_cnt_day,year,month,day,week_day,no_inflation_price,corrected_item_id,item_name,main_category,corrected_shop_id,city
0,2013-01-02,0,999.00,1.0,2013,1,2,2,989.402793,22154,ЯВЛЕНИЕ 2012 (BD),Кино,59,Ярославль
1,2013-01-03,0,899.00,1.0,2013,1,3,3,890.363474,2552,DEEP PURPLE The House Of Blue Light LP,Музыка,25,Москва
2,2013-01-05,0,899.00,-1.0,2013,1,5,5,890.363474,2552,DEEP PURPLE The House Of Blue Light LP,Музыка,25,Москва
3,2013-01-06,0,1709.05,1.0,2013,1,6,6,1692.631475,2554,DEEP PURPLE Who Do You Think We Are LP,Музыка,25,Москва
4,2013-01-15,0,1099.00,1.0,2013,1,15,1,1088.442112,2555,DEEP PURPLE 30 Very Best Of 2CD (Фирм.),Музыка,25,Москва


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2935832 entries, 0 to 2935842
Data columns (total 14 columns):
 #   Column              Dtype         
---  ------              -----         
 0   date                datetime64[ns]
 1   date_block_num      int64         
 2   item_price          float64       
 3   item_cnt_day        float64       
 4   year                int32         
 5   month               int32         
 6   day                 int32         
 7   week_day            int32         
 8   no_inflation_price  float64       
 9   corrected_item_id   int64         
 10  item_name           object        
 11  main_category       object        
 12  corrected_shop_id   int64         
 13  city                object        
dtypes: datetime64[ns](1), float64(3), int32(4), int64(3), object(3)
memory usage: 291.2+ MB


In [14]:
df.isna().sum()

date                  0
date_block_num        0
item_price            0
item_cnt_day          0
year                  0
month                 0
day                   0
week_day              0
no_inflation_price    0
corrected_item_id     0
item_name             0
main_category         0
corrected_shop_id     0
city                  0
dtype: int64

## 9. Preparing the Submission Dataset

The submission dataset is based on the provided `test.csv` file and receives the same identifier corrections that were applied during training.

Specifically, the pipeline:

- loads the competition test set;
- applies corrected item identifiers;
- applies corrected shop identifiers;

In [15]:
for_submission = ForSubmissionLoader(raw_data_path=raw_path, cache_path=processed_path, shops=shops, items=items).load()

for_submission.head()

,ID,shop_id,item_id,corrected_item_id,corrected_shop_id
0,0,5,5037,5037,5
1,1,5,5320,5320,5
2,2,5,5233,5233,5
3,3,5,5232,5232,5
4,4,5,5268,5268,5


In [16]:
for_submission.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 214200 entries, 0 to 214199
Data columns (total 5 columns):
 #   Column             Non-Null Count   Dtype
---  ------             --------------   -----
 0   ID                 214200 non-null  int64
 1   shop_id            214200 non-null  int64
 2   item_id            214200 non-null  int64
 3   corrected_item_id  214200 non-null  int64
 4   corrected_shop_id  214200 non-null  int64
dtypes: int64(5)
memory usage: 8.2 MB
